## Install Required Libraries

In [1]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pandas
!pip install numpy
!pip install transformers
!pip install torch

## Import Libraries

In [2]:
import pandas as pd
import numpy as np
import faiss
import random
from sentence_transformers import SentenceTransformer

## Load Dataset

In [3]:
data = pd.read_csv('MLBot_dataset.csv',encoding='latin1')

print("Dataset loaded successfully")
print("Total rows:", len(data))

Dataset loaded successfully
Total rows: 3413


In [4]:
data.head()

,Questions,Answers
0,What is machine learning?,Machine Learning (ML) is a subfield of artific...
1,How does machine learning differ from traditio...,"In traditional programming, developers write e..."
2,What are the main types of machine learning?,The three primary paradigms of machine learnin...
3,What is supervised learning?,Supervised learning is a type of machine learn...
4,What is unsupervised learning?,Unsupervised learning involves training a mode...


## Basic NLP Cleaning

In [5]:
data["Questions"] = data["Questions"].str.strip()
data["Answers"] = data["Answers"].str.strip()

print("Dataset cleaned.")

Dataset cleaned.


## Prepare Questions & Answers list

In [6]:
Questions = data["Questions"].tolist()
Answers = data["Answers"].tolist()

print("Example question:", Questions[0])
print("Example answer:", Answers[0])

Example question: What is machine learning?
Example answer: Machine Learning (ML) is a subfield of artificial intelligence (AI) focused on creating systems that can learn patterns from data and make predictions or decisions without being explicitly programmed for every task. Unlike traditional programming, where the logic is manually coded, ML systems improve automatically as they are exposed to more data. ML is widely used in applications such as image recognition, natural language processing, fraud detection, recommendation systems, and autonomous vehicles. The core idea is to build models that generalize from past experiences (data) to make predictions on new, unseen data.


## Load Sentence Transformer Model

In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence Transformer model loaded.") 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence Transformer model loaded.


## Generate Embeddings

In [8]:
embeddings = model.encode(Questions)

print("Embeddings created.")
print("Embedding shape:", embeddings.shape)

Embeddings created.
Embedding shape: (3413, 384)


## Build FAISS Vector Database

In [9]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("FAISS index created.")
print("Total vectors stored:", index.ntotal)

FAISS index created.
Total vectors stored: 3413


## Chatbot Logic

In [10]:
def chatbot(question):

    query_embedding = model.encode([question])

    D, I = index.search(query_embedding, k=1)

    similarity_score = D[0][0]

    if similarity_score > 1.5:
        return "Hmm 🤔 I'm not sure about that yet, but I'll definitely try to learn about it!"

    answer = Answers[I[0][0]]

    return answer

## MLBot Greeting

In [11]:
greetings = [
    "Hey there! 👋 I'm MLBot, your friendly AI buddy.",
    "Hello! I'm MLBot. I'm happy to chat with you today!",
    "Hi! I'm MLBot 🤖 Nice to meet you!"
]

print(random.choice(greetings))

name = input("What's your name? ")

print(f"\nNice to meet you {name}! 😊")
print("Ask me anything and I'll try my best to help you.\n")

Hey there! 👋 I'm MLBot, your friendly AI buddy.


What's your name?  Mayank



Nice to meet you Mayank! 😊
Ask me anything and I'll try my best to help you.



## Conversation Loop

In [12]:
while True:

    user_input = input(f"{name}: ")

    if user_input.lower() in ["bye", "exit", "quit"]:
        print("\nMLBot: It was great talking with you! Have a wonderful day 😊")
        break

    response = chatbot(user_input)

    print("\nMLBot:", response)

Mayank:  What is Numpy?



MLBot: NumPy is a Python library used for numerical computing. It provides support for large multi-dimensional arrays and matrices along with mathematical functions to operate on them.


Mayank:  bye



MLBot: It was great talking with you! Have a wonderful day 😊
